# BaytZichlash o‘rniga SBZ qo‘llanilgan 1-algoritmning kriptotahlili

To‘rtta usul: chiziqli, differensial, algebraik va integral. Variantlar: SBZ-256 va SBZ-128
(Aralash, TuzilmaKalit va raund funksiyasi standartdagidek).

Daftar mustaqil ishlaydi: modullar 0-hujayrada yuklanadi, faqat `numpy` kerak.
`HAJM` koeffitsiyenti tanlanmalar hajmini boshqaradi (1 — tez, 4–5 — hisobotdagi hajmga yaqin).
`OGIR = True` qilinsa, to‘liq xesh uchun 2¹⁶ to‘plamli integral sinov ham bajariladi (har bir tajriba bir necha daqiqa).

In [1]:
# 0-hujayra. Dastur modullarini yuklash (daftar mustaqil ishlaydi).
# Agar modullar shu papkada bo'lsa, ular ishlatiladi; bo'lmasa quyidagi matndan xotirada yaratiladi.
import sys, types, importlib.util
_MODULLAR = {
    'algoritm': '"""\nO‘zMSt 285:2024, 1-algoritm: vektorlashtirilgan realizatsiya.\nVariantlar: p = 256 yoki p = 16; yutish o‘zgartirishi bz = \'std\' (BaytZichlash) yoki \'sbz\' (S(h ⊕ S(n))).\nBarcha funksiyalar (N, 32) o‘lchamli numpy massivlar ustida ishlaydi: bir vaqtda N ta mustaqil hisob.\n"""\nimport numpy as np\n\n# B ilova (majburiy): 1x256 chiziqli zichlash massivi (satr — yuqori yarim bayt)\nZICHLASH = bytes.fromhex(\n    "050d060b010a0f0800040709020c030e"\n    "0807020e0f030b06010c0d0a05040900"\n    "0e020d040c07010b06090005030a080f"\n    "000e090c030d07040f0605010b020a08"\n    "030a0702040c09010e0d0f0800050b06"\n    "02030108000e05090c0b06070a0f0d04"\n    "0a040e0f090508020b0001030c06070d"\n    "0b090a0106040d0f03050e000807020c"\n    "010003070d0b0a0c090e04060f080502"\n    "04080b090e0602050a030c0f070d0001"\n    "090c0f0002010e0a05080b0d04030607"\n    "060b080d07090003040f0a020e010c05"\n    "0f0100050a0803070d02090c060e040b"\n    "0c05040a0b02060d0807030e01000f09"\n    "070f0c060500040e020a080b0d090103"\n    "0d060503080f0c0007010204090b0e0a")\nZ = np.array(list(ZICHLASH), dtype=np.int64)\ndef aes_sbox():\n    s=[0]*256; p=q=1\n    while True:\n        p ^= (p<<1) ^ (0x1b if p&0x80 else 0); p&=0xff\n        q ^= q<<1; q ^= q<<2; q ^= q<<4; q&=0xff\n        if q&0x80: q^=0x09\n        x=q ^ ((q<<1|q>>7)&0xff) ^ ((q<<2|q>>6)&0xff) ^ ((q<<3|q>>5)&0xff) ^ ((q<<4|q>>4)&0xff)\n        s[p]=x^0x63\n        if p==1: break\n    s[0]=0x63; return np.array(s,dtype=np.int64)\nS=aes_sbox()\nclass Cfg:\n    def __init__(s,p=256,bz=\'sbz\'):\n        s.p=p; s.bz=bz\n        s.INV=np.zeros(p,dtype=np.int64)\n        for x in range(1,p,2): s.INV[x]=pow(x,-1,p)\ndef prododd(a,p):\n    b=np.where(a%2==0,a+1,a)%p; r=np.ones(a.shape[0],dtype=np.int64)\n    for i in range(a.shape[1]): r=(r*b[:,i])%p\n    return r\ndef dK(P,Jo,p): return np.where((P==(Jo+1)%p)|(P==(-(Jo+1))%p),10,0)\ndef step(x,A,B,R,c):\n    d=(B+R*x)%c.p; return (-A*x*c.INV[d])%c.p\ndef qosh(h,hn,R,c):\n    p=c.p; R=R[:,None]\n    J1h=h.sum(1)%p; J1hn=hn.sum(1)%p\n    Ph=prododd(h,p); Phn=prododd(hn,p)\n    K1h=(Ph+dK(Ph,J1hn,p))%p; K1hn=(Phn+dK(Phn,J1h,p))%p\n    dAh=np.where(J1hn%2==1,0,1); dAhn=np.where(J1h%2==1,0,1)\n    A1h=(J1hn[:,None]+2*hn+dAh[:,None])%p; A1hn=(J1h[:,None]+2*h+dAhn[:,None])%p\n    h1=step(h,A1h,(K1h[:,None]-2*h)%p,R,c); hn1=step(hn,A1hn,(K1hn[:,None]-2*hn)%p,R,c)\n    J2h=h1.sum(1)%p; J2hn=hn1.sum(1)%p\n    P2h=prododd(h1,p); P2hn=prododd(hn1,p)\n    K2h=(P2h+dK(P2h,J2hn,p))%p; K2hn=(P2hn+dK(P2hn,J2h,p))%p\n    dA2h=np.where((J1hn+J2hn)%2==1,0,1); dA2hn=np.where((J1h+J2h)%2==1,0,1)\n    A2h=(J1hn[:,None]+J2hn[:,None]+2*hn1+dA2h[:,None])%p; A2h=np.where(A2h==0,1,A2h)\n    A2hn=(J1h[:,None]+J2h[:,None]+2*h1+dA2hn[:,None])%p; A2hn=np.where(A2hn==0,1,A2hn)\n    B2h=((K1h*K2h)[:,None]-2*h1)%p; B2hn=((K1hn*K2hn)[:,None]-2*hn1)%p\n    h2=step(h1,A2h,B2h,R,c); h2=np.where(h2==0,(A2h+B2h)%p,h2)\n    hn2=step(hn1,A2hn,B2hn,R,c); hn2=np.where(hn2==0,(A2hn+B2hn)%p,hn2)\n    return step(h2,A2h,B2h,R,c), step(hn2,A2hn,B2hn,R,c)\ndef bzf(h,hn,c):\n    if c.p==256:\n        if c.bz==\'std\': return (Z[h]<<4)|Z[hn]\n        return S[h^S[hn]]\n    if c.bz==\'std\': return h^hn\n    H=16*h[:,0::2]+h[:,1::2]; N=16*hn[:,0::2]+hn[:,1::2]; Q=S[H^S[N]]\n    out=np.empty_like(h); out[:,0::2]=Q>>4; out[:,1::2]=Q&15; return out\ndef dia(H,K,p):  # H,K: (N,4,4)\n    sk=K.sum(1)            # sum_i K[i,u]  -> (N,4)\n    sh=H.sum(1)            # sum_i H[i,u]\n    R=np.empty_like(H)\n    for s in range(4):\n        for u in range(4):\n            if s==u:\n                v=H[:,u,u]*sk[:,u]-sum(H[:,i,i]*K[:,i,u] for i in range(4) if i!=u)\n            else:\n                v=H[:,s,u]*sk[:,u]+K[:,s,u]*sh[:,u]-sum(H[:,s,i]*K[:,i,u] for i in range(4) if i not in (s,u))\n            R[:,s,u]=v%p\n    return R\ndef aralash(h,ke,c):\n    H=h.reshape(-1,4,8); K=ke.reshape(-1,4,8)\n    L=dia(H[:,:,:4],K[:,:,:4],c.p); Rr=dia(H[:,:,4:],K[:,:,4:],c.p)\n    return np.concatenate([L,Rr],axis=2).reshape(-1,32)\ndef _shift(h,rows,cols):\n    m=h.reshape(-1,4,8).copy()\n    for s in range(4): m[:,s,:]=np.roll(m[:,s,:],rows[s],axis=1)\n    for u,n in cols.items(): m[:,:,u]=np.roll(m[:,:,u],n,axis=1)\n    return m.reshape(-1,32)\ndef surholat(h): return _shift(h,[1,2,3,4],{0:1,4:1,1:2,5:2,2:3,6:3})\ndef surkalit(ke): return _shift(ke,[4,3,2,1],{1:-3,5:-3,2:-2,6:-2,3:-1,7:-1})\ndef tuz(ke,k,c):\n    p=c.p\n    return np.where((ke==0)&(k%2==0),2,np.where((ke!=0)&(ke%2==k%2),ke,(ke-1)%p))\ndef iteration(holat,blok,ke,k,c,trace=None,rounds=10):\n    """Bitta tashqi iteratsiya (i): Qo\'sh+BZ, so\'ng \'rounds\' ta bosqich."""\n    R=k[:,0]\n    def T(name,j,h,kk):\n        if trace is not None: trace.append((name,j,h.copy(),kk.copy()))\n    holat,hn=qosh(holat,blok,R,c); T(\'Qosh0\',0,holat,ke)\n    holat=bzf(holat,hn,c); T(\'BZ0\',0,holat,ke)\n    for j in range(1,rounds+1):\n        holat=aralash(holat,ke,c); T(\'Aralash\',j,holat,ke)\n        holat,ke=qosh(holat,ke,R,c); T(\'Qosh1\',j,holat,ke)\n        holat=surholat(holat); ke=surkalit(ke); T(\'Sur\',j,holat,ke)\n        if j==8:\n            holat,hn=qosh(holat,blok,R,c); T(\'Qosh_inj8\',j,holat,ke); holat=bzf(holat,hn,c); T(\'BZ8\',j,holat,ke)\n        else:\n            holat,ke=qosh(holat,ke,R,c); T(\'Qosh2\',j,holat,ke)\n        ke=tuz(ke,k,c); T(\'Tuz\',j,holat,ke)\n        if j==9:\n            holat,hn=qosh(holat,blok,R,c); T(\'Qosh_inj9\',j,holat,ke); holat=bzf(holat,hn,c); T(\'BZ9\',j,holat,ke)\n    return holat,ke\ndef to_units(b,p):\n    b=np.asarray(b,dtype=np.int64)\n    if p==256: return b\n    out=np.empty(b.shape[:-1]+(b.shape[-1]*2,),dtype=np.int64); out[...,0::2]=b>>4; out[...,1::2]=b&15; return out\ndef hash_msgs(msgs,keys,c):\n    """msgs: (N,L) bayt, L - blok uzunligiga karrali; keys: (N,32 yoki 16)."""\n    return _hash_core(msgs,keys,c,bits=msgs.shape[1]*8)\ndef _hash_core(msgs,keys,c,bits):\n    N,L=msgs.shape; bl=32 if c.p==256 else 16\n    nb=L//bl\n    blocks=[msgs[:,i*bl:(i+1)*bl] for i in range(nb)]\n    ny=[0]*N\n    nyb=np.zeros((N,bl),dtype=np.int64)\n    for n in range(N):\n        v=sum(int.from_bytes(bytes(b[n].tolist()),\'big\') for b in blocks)%(2**(8*bl))\n        nyb[n]=list(v.to_bytes(bl,\'big\'))\n    lb=np.tile(np.array(list(bits.to_bytes(bl,\'big\'))),(N,1))\n    kir=[to_units(b,c.p) for b in blocks]+[to_units(lb,c.p),to_units(nyb,c.p)]\n    k=to_units(keys,c.p); ke=k.copy(); holat=kir[0]\n    for i in range(1,len(kir)): holat,ke=iteration(holat,kir[i],ke,k,c)\n    return holat\n\n\n# ------------------------------------------------------------------ nazorat qiymatlari\nKAT_KALIT = bytes.fromhex("c86ed4b2756df1cd6b6eca81d441809221ab81bba0c7c15044be8feb934f95a2")\nKAT_XABAR = b"funksiyaxeshirovaniyainformasionnayatexnologiyakriptograficheska"\nKAT = {\n    (256, \'std\'): "55394f2d84fccd4d8d01d8fef756cd657ebecc3f96f81e5207767447de66e2b6",\n    (256, \'sbz\'): "42f97daaa8e7010bc629ca695eaa77df7ae7443a0c3ae78d205c2e025d256c96",\n    (16, \'std\'):  "e9eaccdd4ccce2d229228c864989eb9f",\n    (16, \'sbz\'):  "bd745e2ed572ea4246ed68482ef46c5e",\n}\n\ndef xesh(xabar: bytes, kalit: bytes, p=256, bz=\'sbz\') -> bytes:\n    """Bitta xabar uchun xesh qiymat (xabar uzunligi blok uzunligiga karrali bo‘lishi shart emas)."""\n    bl = 32 if p == 256 else 16\n    m = xabar + b"\\x00" * (-len(xabar) % bl) if xabar else b"\\x00" * bl\n    # uzunlik bloki asl xabar uzunligi bo‘yicha hisoblanadi\n    c = Cfg(p, bz)\n    msgs = np.array([list(m)]); keys = np.array([list(kalit[:bl])])\n    out = _hash_core(msgs, keys, c, bits=len(xabar) * 8)[0]\n    if p == 16: out = [out[2*i] << 4 | out[2*i+1] for i in range(16)]\n    return bytes(int(x) for x in out)\n\ndef nazorat():\n    """A ilova va SBZ materialidagi nazorat qiymatlarini tekshiradi."""\n    ok = True\n    for (p, bz), kut in KAT.items():\n        r = xesh(KAT_XABAR, KAT_KALIT, p, bz).hex()\n        print(f"  p={p:3d}, {bz}: {\'MOS\' if r == kut else \'MOS EMAS\'}")\n        ok &= (r == kut)\n    return ok\n\nif __name__ == "__main__":\n    print("Nazorat qiymatlari:"); print("Hammasi mos" if nazorat() else "XATO")\n',
    'komponentlar': '"""Kriptotahlil uchun umumiy yordamchi funksiyalar."""\nimport itertools\nimport numpy as np\nfrom algoritm import Cfg, aralash, surholat\n\nPAR8 = np.array([bin(i).count(\'1\') & 1 for i in range(256)])\nX256 = np.arange(256)\n\ndef kalitlar(rng, N, p):\n    """Tasodifiy kalitlar: R (1-element) va 2-element juft (5.1.2-band talabi)."""\n    if p == 256:\n        k = rng.integers(0, 256, (N, 32)); k[:, 0] &= 0xFE; k[:, 1] &= 0xFE\n    else:\n        k = rng.integers(0, 16, (N, 32)); k[:, 0] &= 0xE; k[:, 1] &= 0xE\n    return k\n\ndef wht(f):\n    """Walsh–Adamar o‘zgartirishi (f — ±1 vektor)."""\n    a = f.astype(np.int64).copy(); h = 1; n = len(a)\n    while h < n:\n        a = a.reshape(-1, 2, h); a = np.stack([a[:, 0] + a[:, 1], a[:, 0] - a[:, 1]], 1).reshape(-1); h *= 2\n    return a\n\ndef lat(F, nout):\n    """8 bitli kirishli F uchun W[β][α] = Σ (−1)^(β·F(x) ⊕ α·x); korrelyatsiya c = W/256."""\n    W = np.zeros((1 << nout, 256), dtype=np.int64)\n    for b in range(1 << nout):\n        W[b] = wht(1 - 2 * PAR8[F & b])\n    return W\n\ndef ddt(F, nout):\n    """Ayirmalar matritsasi DDT[Δx][Δy] (8 bitli kirish)."""\n    return np.array([np.bincount(F ^ F[X256 ^ d], minlength=nout) for d in range(256)])\n\ndef anf_deg(fbits, nv):\n    """Boolean funksiyaning ANF darajasi (Möbius o‘zgartirishi)."""\n    a = fbits.astype(np.uint8).copy(); h = 1; N = 1 << nv\n    while h < N:\n        a = a.reshape(-1, 2, h); a[:, 1] ^= a[:, 0]; a = a.reshape(-1); h *= 2\n    wt = np.array([bin(i).count(\'1\') for i in range(N)]); nz = wt[a == 1]\n    return int(nz.max()) if len(nz) else -1\n\ndef bits(v, n): return (v[:, None] >> np.arange(n)) & 1\n\ndef implicit_count(points, deg):\n    """deg darajali chiziqli bog‘liq bo‘lmagan implitsit tenglamalar soni (monomlar matritsasi rangi orqali)."""\n    npts, nv = points.shape\n    mons = [()] + [m for d in range(1, deg + 1) for m in itertools.combinations(range(nv), d)]\n    words = (npts + 63) // 64; cols = []\n    for m in mons:\n        v = np.ones(npts, dtype=np.uint8)\n        for i in m: v &= points[:, i].astype(np.uint8)\n        pad = np.zeros(words * 64, dtype=np.uint8); pad[:npts] = v\n        cols.append(np.packbits(pad).view(np.uint64))\n    A = np.array(cols); rank = 0; n = A.shape[0]\n    for w in range(A.shape[1]):\n        for b in range(64):\n            if rank >= n: break\n            mask = np.uint64(1) << np.uint64(b)\n            idx = np.nonzero((A[rank:, w] & mask) != 0)[0]\n            if len(idx) == 0: continue\n            piv = rank + idx[0]\n            if piv != rank: A[[rank, piv]] = A[[piv, rank]]\n            rows = np.nonzero((A[:, w] & mask) != 0)[0]; rows = rows[rows != rank]\n            A[rows] ^= A[rank]; rank += 1\n    return len(mons) - rank, len(mons)\n\ndef qosh_elem(A1, B1, A2, B2, R, p=256):\n    """Qo‘sh o‘zgartirishining bitta elementi (parametrlar belgilangan): x → x^(−3)."""\n    x = np.arange(p); INV = np.zeros(p, dtype=np.int64)\n    for t in range(1, p, 2): INV[t] = pow(t, -1, p)\n    s = lambda x, A, B: (-A * x * INV[(B + R * x) % p]) % p\n    y1 = s(x, A1, B1); y2 = s(y1, A2, B2); y2 = np.where(y2 == 0, (A2 + B2) % p, y2)\n    return s(y2, A2, B2)\n\ndef tasodifiy_param(rng):\n    A1, B1, A2, B2 = [int(rng.integers(0, 128)) * 2 + 1 for _ in range(4)]\n    return A1, B1, A2, B2, int(rng.integers(0, 128)) * 2\n\ndef rank2(A):\n    A = A.copy() % 2; r = 0\n    for col in range(A.shape[1]):\n        piv = [i for i in range(r, A.shape[0]) if A[i, col]]\n        if not piv: continue\n        A[[r, piv[0]]] = A[[piv[0], r]]\n        for i in range(A.shape[0]):\n            if i != r and A[i, col]: A[i] ^= A[r]\n        r += 1\n    return r\n\n# 0-qatlam matritsalari: SurHolat (P) va Aralash (M(k0))\n_C2 = Cfg(2, \'std\')\n_perm = surholat(np.arange(32)[None, :].copy())[0]\nP_SUR = np.zeros((32, 32), dtype=np.int64); P_SUR[np.arange(32), _perm] = 1\n\ndef M_aralash(k):\n    """h0\' = M(k0)·h0 (mod 2) matritsasi."""\n    E = np.eye(32, dtype=np.int64); K = np.tile(np.asarray(k) % 2, (32, 1))\n    return (aralash(E, K, _C2).T) % 2\n\ndef L_raund(k):\n    """Yutishsiz raundning 0-qatlam matritsasi L = P·M(k0)."""\n    return (P_SUR @ M_aralash(k)) % 2\n',
    'differensial': '"""\n2-bo‘lim. Differensial kriptotahlil.\n  1) ayirmalar matritsalari: S, Z, SBZ, Qo‘sh elementi;\n  2) orttirmalarni raundlar va har bir akslantirish bo‘yicha kuzatish:\n       D0a/D0b — yutishdan keyingi holatdagi 1 / 2 baytli MSB ayirmasi,\n       D1 — iteratsiya boshidagi holatdagi 1 baytli MSB ayirmasi,\n       D2 — kirish blokidagi 2 baytli MSB ayirmasi, D3 — kirish blokidagi 1 baytli MSB ayirmasi;\n  3) Aralash yadrosi: nol va teng ustunli kalitlar ulushi.\nNatija: natijalar/differensial.json\n"""\nimport argparse, collections, json, os\nimport numpy as np\nfrom algoritm import S, Z, Cfg, iteration, aralash, qosh, surholat, surkalit, tuz, bzf\nfrom komponentlar import ddt, qosh_elem, tasodifiy_param, kalitlar, M_aralash\n\ndef komponentlar_tahlili(rng, trials=200):\n    R = {}; DS = ddt(S, 256); DZ = ddt(Z, 16); x = np.arange(256)\n    dist = lambda D: {int(a): int(b) for a, b in zip(*np.unique(D[1:], return_counts=True))}\n    R[\'S_max\'] = int(DS[1:].max()); R[\'S_taqsimot\'] = dist(DS)\n    R[\'S_bolagi\'] = [[int(DS[d, y]) for y in range(8)] for d in range(1, 9)]\n    R[\'Z_max\'] = int(DZ[1:].max()); i = np.unravel_index(DZ[1:].argmax(), DZ[1:].shape)\n    R[\'Z_max_joyi\'] = (hex(int(i[0]) + 1), hex(int(i[1]))); R[\'Z_taqsimot\'] = dist(DZ)\n    R[\'Z_sonishi_mumkin_ayirmalar\'] = int((DZ[1:, 0] > 0).sum())\n    for d in (0x80, 0x01, 0x96): R[f\'Z_satr_{d:02x}\'] = DZ[d].tolist()\n    def sbz(dh, dn):\n        if dn == 0: return DS[dh] / 256.0\n        return sum(DS[dn, d] * DS[dh ^ d] for d in range(256) if DS[dn, d]) / 65536.0\n    R[\'SBZ_dh0_dn80_max\'] = float(sbz(0, 0x80).max()); R[\'SBZ_dh80_dn0_max\'] = float(sbz(0x80, 0).max())\n    R[\'SBZ_dh0_dn80_sonish\'] = float(sbz(0, 0x80)[0]); R[\'SBZ_dh80_dn80_sonish\'] = float(sbz(0x80, 0x80)[0])\n    R[\'SBZ_80dan80ga\'] = float((DS[0x80] * DS[:, 0x80]).sum() / 65536)\n    mx, p80, pj = [], [], {j: [] for j in range(8)}\n    for _ in range(trials):\n        f = qosh_elem(*tasodifiy_param(rng)); D = ddt(f, 256)\n        mx.append(int(D[1:].max())); p80.append(D[0x80, 0x80] / 256)\n        for j in range(8): pj[j].append(D[1 << j].max() / 256)\n    R[\'Qosh_max\'] = int(max(mx)); R[\'Qosh_p(80->80)\'] = float(np.mean(p80))\n    R[\'Qosh_2^j_eng_ehtimolli\'] = [round(float(np.mean(pj[j])), 4) for j in range(8)]\n    return R\n\ndef raundlar_yutishsiz(holat, blok, ke, k, c, trace):\n    """Yutishdan keyingi holatdan boshlab 1–10-raundlar (0-nuqtadagi yutishsiz)."""\n    R = k[:, 0]\n    def T(nm, j): trace.append((nm, j, holat.copy(), ke.copy()))\n    for j in range(1, 11):\n        holat = aralash(holat, ke, c); T(\'Aralash\', j)\n        holat, ke = qosh(holat, ke, R, c); T(\'Qosh1\', j)\n        holat = surholat(holat); ke = surkalit(ke); T(\'Sur\', j)\n        if j == 8:\n            holat, hn = qosh(holat, blok, R, c); T(\'Qosh_inj8\', j); holat = bzf(holat, hn, c); T(\'BZ8\', j)\n        else:\n            holat, ke = qosh(holat, ke, R, c); T(\'Qosh2\', j)\n        ke = tuz(ke, k, c); T(\'Tuz\', j)\n        if j == 9:\n            holat, hn = qosh(holat, blok, R, c); T(\'Qosh_inj9\', j); holat = bzf(holat, hn, c); T(\'BZ9\', j)\n\ndef hw(a, p): return sum(((a >> i) & 1) for i in range(8 if p == 256 else 4)).sum(1)\n\ndef kuzatish(rng, N=2000, holatlar=(\'D1\', \'D2\', \'D3\')):\n    R = {}\n    for p, bz in [(256, \'std\'), (256, \'sbz\'), (16, \'std\'), (16, \'sbz\')]:\n        c = Cfg(p, bz); msb = p // 2\n        for case in holatlar:\n            k = kalitlar(rng, N, p); h = rng.integers(0, p, (N, 32)); bl = rng.integers(0, p, (N, 32))\n            h2, bl2 = h.copy(), bl.copy()\n            if case in (\'D0a\', \'D1\'): h2[:, 0] ^= msb\n            elif case == \'D0b\': h2[:, 0] ^= msb; h2[:, 9] ^= msb\n            elif case == \'D2\': bl2[:, 0] ^= msb; bl2[:, 9] ^= msb\n            else: bl2[:, 0] ^= msb\n            t1, t2 = [], []\n            if case.startswith(\'D0\'):\n                raundlar_yutishsiz(h, bl, k.copy(), k, c, t1); raundlar_yutishsiz(h2, bl, k.copy(), k, c, t2)\n            else:\n                iteration(h, bl, k.copy(), k, c, trace=t1); iteration(h2, bl2, k.copy(), k, c, trace=t2)\n            rows = []\n            for (nm, j, a, ka), (_, _, b, kb) in zip(t1, t2):\n                d, dk = a ^ b, ka ^ kb; act = (d != 0).sum(1)\n                msbonly = (((d == 0) | (d == msb)).all(1) & ((dk == 0) | (dk == msb)).all(1)).mean()\n                top = collections.Counter(map(bytes, np.concatenate([d, dk], 1).astype(np.uint8))).most_common(1)[0][1] / N\n                rows.append(dict(amal=nm, raund=j, faol_holat=round(float(act.mean()), 2), faol_holat_min=int(act.min()),\n                                 faol_ke=round(float((dk != 0).sum(1).mean()), 2), P_MSB_izi=round(float(msbonly), 4),\n                                 eng_ehtimolli_ayirma=round(top, 4), xemming=round(float(hw(d, p).mean()), 1),\n                                 nol_ayirma=round(float(((act == 0) & ((dk != 0).sum(1) == 0)).mean()), 4)))\n            R[f\'{p}_{bz}_{case}\'] = rows\n        print(f"  p={p}, {bz}: tayyor")\n    return R\n\ndef yadro(rng, nkeys=300):\n    zc = eq = 0\n    for _ in range(nkeys):\n        M = M_aralash(kalitlar(rng, 1, 256)[0]); cols = [tuple(M[:, i]) for i in range(32)]\n        zc += any(sum(cc) == 0 for cc in cols); eq += len(set(cols)) < 32\n    return {\'nol_ustunli_kalitlar\': zc / nkeys, \'teng_ustunli_kalitlar\': eq / nkeys}\n\nif __name__ == "__main__":\n    ap = argparse.ArgumentParser(); ap.add_argument(\'--tez\', action=\'store_true\')\n    a = ap.parse_args(); s = 0.15 if a.tez else 1.0\n    os.makedirs(\'natijalar\', exist_ok=True); out = {}\n    print("1. Ayirmalar matritsalari..."); out[\'komponentlar\'] = komponentlar_tahlili(np.random.default_rng(11), int(200 * s))\n    print("2. Orttirmalarni kuzatish...")\n    out[\'kuzatish\'] = kuzatish(np.random.default_rng(5), int(2000 * s), (\'D1\', \'D2\', \'D3\'))\n    out[\'kuzatish\'].update(kuzatish(np.random.default_rng(9), int(3000 * s), (\'D0a\', \'D0b\')))\n    print("3. Aralash yadrosi..."); out[\'yadro\'] = yadro(np.random.default_rng(31), int(300 * s))\n    json.dump(out, open(\'natijalar/differensial.json\', \'w\'), indent=1, ensure_ascii=False)\n    print("Saqlandi: natijalar/differensial.json")\n',
    'algebraik': '"""\n3-bo‘lim. Algebraik kriptotahlil.\n  1) implitsit tenglamalar soni va ANF darajalari: S, Z, SBZ, BaytZichlash-256, Qo‘sh elementi;\n  2) kub testlari: r = 0…3 raunddan keyin daraja o‘sishi;\n  3) hujum namoyishi: chiqish 0-qatlamini xabar 0-qatlamining affin funksiyasi sifatida tiklash;\n  4) bit darajasidagi tenglamalar soni (baho modeli) va linearizatsiya murakkabligi.\nNatija: natijalar/algebraik.json\n"""\nimport argparse, json, math, os\nimport numpy as np\nfrom algoritm import S, Z, Cfg, iteration, hash_msgs, to_units\nfrom komponentlar import implicit_count, anf_deg, bits, qosh_elem, tasodifiy_param, kalitlar\n\ndef tenglamalar_va_daraja(rng, trials=50):\n    R = {}; x = np.arange(256)\n    R[\'S_kvadrat\'] = implicit_count(np.concatenate([bits(x, 8), bits(S[x], 8)], 1), 2)\n    R[\'Z_kvadrat\'] = implicit_count(np.concatenate([bits(x, 8), bits(Z[x], 4)], 1), 2)\n    R[\'Z_kub\'] = implicit_count(np.concatenate([bits(x, 8), bits(Z[x], 4)], 1), 3)\n    f = qosh_elem(*tasodifiy_param(rng))\n    R[\'Qosh_kvadrat\'] = implicit_count(np.concatenate([bits(x, 8), bits(f, 8)], 1), 2)\n    R[\'Qosh_kub\'] = implicit_count(np.concatenate([bits(x, 8), bits(f, 8)], 1), 3)\n    hh = np.arange(65536) >> 8; nn = np.arange(65536) & 255\n    y = S[hh ^ S[nn]]; yb = (Z[hh] << 4) | Z[nn]\n    R[\'SBZ_kvadrat\'] = implicit_count(np.concatenate([bits(hh, 8), bits(nn, 8), bits(y, 8)], 1), 2)\n    R[\'BZ256_kvadrat\'] = implicit_count(np.concatenate([bits(hh, 8), bits(nn, 8), bits(yb, 8)], 1), 2)\n    R[\'S_daraja\'] = [anf_deg((S >> j) & 1, 8) for j in range(8)]\n    R[\'Z_daraja\'] = [anf_deg((Z >> j) & 1, 8) for j in range(4)]\n    R[\'SBZ_daraja\'] = [anf_deg((y >> j) & 1, 16) for j in range(8)]\n    R[\'BZ256_daraja\'] = [anf_deg((yb >> j) & 1, 16) for j in range(8)]\n    D = np.array([[anf_deg((qosh_elem(*tasodifiy_param(rng)) >> j) & 1, 8) for j in range(8)] for _ in range(trials)])\n    R[\'Qosh_daraja_min\'] = D.min(0).tolist(); R[\'Qosh_daraja_mediana\'] = np.median(D, 0).tolist(); R[\'Qosh_daraja_max\'] = D.max(0).tolist()\n    return R\n\ndef kub_testlari(rng, dlar=(2, 4, 6, 8, 10, 12), raundlar=(0, 1, 2, 3), ncubes=4):\n    """Xabarning dastlabki 16 bitidan d ta bit; 4 ta kub bo‘yicha yig‘indisi doim nol bo‘lgan bitlar ulushi."""\n    R = {}\n    for p, bz in [(256, \'std\'), (256, \'sbz\'), (16, \'std\'), (16, \'sbz\')]:\n        c = Cfg(p, bz); ub = 8 if p == 256 else 4\n        for r in raundlar:\n            row = {}\n            for d in dlar:\n                k = kalitlar(rng, 1, p)[0]; h = rng.integers(0, p, 32); bl = rng.integers(0, p, 32); allz = None\n                for _ in range(ncubes):\n                    pos = rng.choice(16, d, replace=False); base = bl.copy(); base[:16 // ub] = rng.integers(0, p, 16 // ub)\n                    N = 1 << d; Bm = np.tile(base, (N, 1))\n                    for t, q in enumerate(pos):\n                        u, b = q // ub, q % ub; Bm[:, u] &= ~(1 << b); Bm[:, u] |= ((np.arange(N) >> t) & 1) << b\n                    K = np.tile(k, (N, 1)); out, _ = iteration(np.tile(h, (N, 1)), Bm, K.copy(), K, c, rounds=r)\n                    s = np.bitwise_xor.reduce(out, axis=0); zb = np.array([(s >> j) & 1 for j in range(ub)]).T == 0\n                    allz = zb if allz is None else (allz & zb)\n                row[str(d)] = {\'barcha_bitlar\': round(float(allz.mean()), 3), \'0-qatlam\': round(float(allz[:, 0].mean()), 3)}\n            R[f\'{p}_{bz}_r{r}\'] = row\n            print(f"  p={p}, {bz}, r={r}: " + ", ".join(f"d={d}:{v[\'barcha_bitlar\']}" for d, v in row.items()))\n    return R\n\ndef gf2_solve(X, Y):\n    A = np.concatenate([X, Y], 1).astype(np.uint8) % 2; m, n = X.shape; r = 0; piv = []\n    for col in range(n):\n        idx = [i for i in range(r, m) if A[i, col]]\n        if not idx: continue\n        A[[r, idx[0]]] = A[[idx[0], r]]\n        for i in range(m):\n            if i != r and A[i, col]: A[i] ^= A[r]\n        piv.append(col); r += 1\n    W = np.zeros((n, Y.shape[1]), dtype=np.uint8)\n    for i, col in enumerate(piv): W[col] = A[i, n:]\n    return W\n\ndef linearizatsiya_hujumi(rng, n_oquv=80, n_sinov=300):\n    R = {}\n    for p, bz in [(16, \'std\'), (16, \'sbz\'), (256, \'std\'), (256, \'sbz\')]:\n        c = Cfg(p, bz); bl = 32 if p == 256 else 16\n        if p == 256: k = rng.integers(0, 256, 32); k[0] &= 0xFE; k[1] &= 0xFE\n        else: k = rng.integers(0, 256, 16); k[0] &= 0xEE\n        H = lambda m: hash_msgs(m, np.tile(k, (len(m), 1)), c)\n        feats = lambda m: np.concatenate([to_units(m, p) % 2, np.ones((len(m), 1), dtype=np.int64)], 1)\n        tr = rng.integers(0, 256, (n_oquv, bl)); te = rng.integers(0, 256, (n_sinov, bl))\n        W = gf2_solve(feats(tr), H(tr) % 2); pred = (feats(te) @ W) % 2; act = H(te) % 2\n        R[f\'{p}_{bz}\'] = {\'toliq_32_bit\': float((pred == act).all(1).mean()), \'bitlar_boyicha\': round(float((pred == act).mean()), 4)}\n        print(f"  p={p}, {bz}: {R[f\'{p}_{bz}\']}")\n    return R\n\ndef tenglamalar_soni():\n    """Bit darajasidagi baho modeli: (o‘zgaruvchilar, tenglamalar, nochiziqli tenglamalar)."""\n    def counts(n, mode):\n        w = 32; ADD = (2*n-1, 2*n-1, n-1); MUL = (n*(n+1)//2 + (n-1)*(2*n-1),) * 2 + (n*(n+1)//2 + (n-1)*(n-1),)\n        add = lambda a, b, f=1: tuple(x + f*y for x, y in zip(a, b)); z = (0, 0, 0); q = z\n        for _ in range(2):\n            q = add(q, ADD, (w-1)*2); q = add(q, MUL, (w-1)*2); q = add(q, (2*n,)*3, 4); q = add(q, MUL, 1)\n            q = add(q, ADD, w*4); q = add(q, MUL, w*9); q = add(q, ADD, w*6); q = add(q, (n,)*3, w)\n        ar = add(add(z, MUL, 128), ADD, 144); tz = add(add(z, ADD, w), (n,)*3, w)\n        if mode == \'sbz\': nb = w if n == 8 else w // 2; bz = (16*nb, 78*nb, 78*nb)\n        elif n == 8: bz = (8*w, 43*2*w + 8*w, 43*2*w)\n        else: bz = (0, n*w, 0)\n        rnd = add(add(ar, q, 2), tz); r0 = add(q, bz)\n        out = dict(qosh=q, aralash=ar, tuzilma=tz, yutish=bz, raund=rnd, r1=add(r0, rnd))\n        out[\'r2\'] = add(out[\'r1\'], rnd); out[\'r3\'] = add(out[\'r2\'], rnd); return out\n    E = {f\'{lab}_{m}\': counts(n, m) for n, lab in [(8, \'256\'), (4, \'128\')] for m in (\'std\', \'sbz\')}\n    lc = lambda nv, d: round(2.8 * math.log2(sum(math.comb(nv, i) for i in range(d + 1))), 1)\n    return E, {f\'{nv}_ozg_d{d}\': lc(nv, d) for nv in (128, 256) for d in (1, 4, 6, 7, 8, 13)}\n\nif __name__ == "__main__":\n    ap = argparse.ArgumentParser(); ap.add_argument(\'--tez\', action=\'store_true\')\n    a = ap.parse_args(); os.makedirs(\'natijalar\', exist_ok=True); out = {}\n    print("1. Implitsit tenglamalar va darajalar..."); out[\'tenglamalar_daraja\'] = tenglamalar_va_daraja(np.random.default_rng(13), 10 if a.tez else 50)\n    print("2. Kub testlari..."); out[\'kub\'] = kub_testlari(np.random.default_rng(21), dlar=(2, 4, 6, 8) if a.tez else (2, 4, 6, 8, 10, 12), raundlar=(0, 1) if a.tez else (0, 1, 2, 3))\n    print("3. Linearizatsiya hujumi..."); out[\'linearizatsiya\'] = linearizatsiya_hujumi(np.random.default_rng(31), 80, 100 if a.tez else 300)\n    E, LC = tenglamalar_soni(); out[\'tenglamalar_soni\'] = E; out[\'linearizatsiya_murakkabligi_log2\'] = LC\n    json.dump(out, open(\'natijalar/algebraik.json\', \'w\'), indent=1, ensure_ascii=False, default=list)\n    print("Saqlandi: natijalar/algebraik.json")\n',
    'integral': '"""\n4-bo‘lim. Integral (square) kriptotahlil — standart algoritm (BaytZichlash) va SBZ varianti.\nTo‘plam xossalari: A (barcha qiymatlar bir martadan), C (doimiy), B (XOR-yig‘indi = 0),\nB7 (XOR-yig‘indining 0–6-bitlari = 0, ya’ni yig‘indi ∈ {00, 80}), U (noma’lum).\n  1) akslantirishlarning tanlangan to‘plamlarga ta’siri (Qo‘sh elementi va massivi, Aralash, Z, XOR);\n  2) raundlar va akslantirishlar bo‘yicha integral xossalarni kuzatish;\n  3) to‘liq xesh uchun integral farqlovchi;\n  4) Standart-128 da T-funksiya xossasi: yuqori bit-qatlam o‘zgarganda quyi qatlamlar saqlanishi.\nNatija: natijalar/integral.json\n"""\nimport argparse, json, os\nimport numpy as np\nfrom algoritm import S, Z, Cfg, iteration, qosh, aralash, surholat, surkalit, tuz, bzf, hash_msgs, to_units\nfrom komponentlar import qosh_elem, tasodifiy_param, kalitlar\n\ndef xossa(V, p):\n    """V: (N, 32) — to‘plam bo‘yicha qiymatlar. Har pozitsiya uchun: \'A\',\'C\',\'B\',\'B7\',\'U\'."""\n    msb = p // 2; out = []\n    s = np.bitwise_xor.reduce(V, axis=0)\n    for i in range(V.shape[1]):\n        col = V[:, i]; nd = len(np.unique(col))\n        if nd == 1: out.append(\'C\')\n        elif nd == V.shape[0] and V.shape[0] == p: out.append(\'A\')\n        elif s[i] == 0: out.append(\'B\')\n        elif s[i] == msb: out.append(\'B7\')\n        else: out.append(\'U\')\n    return out\n\ndef sana(xs):\n    return {t: xs.count(t) for t in (\'A\', \'C\', \'B\', \'B7\', \'U\')}\n\ndef komponentlar(rng, trials=200, bz=\'std\'):\n    R = {}; x = np.arange(256)\n    # Qo\'sh elementi: 256 kirish -> chiqish to\'plami\n    nd, sm = [], []\n    for _ in range(trials):\n        f = qosh_elem(*tasodifiy_param(rng)); nd.append(len(np.unique(f))); sm.append(int(np.bitwise_xor.reduce(f)))\n    R[\'Qosh_element_turli_qiymatlar\'] = {str(k): nd.count(k) for k in sorted(set(nd))}\n    R[\'Qosh_element_xor_yigindi\'] = {\'00\': sm.count(0), \'80\': sm.count(0x80), \'boshqa\': trials - sm.count(0) - sm.count(0x80)}\n    # Z: faol kirish -> chiqish\n    R[\'Z_xor_yigindi\'] = int(np.bitwise_xor.reduce(Z)); R[\'Z_har_qiymat_soni\'] = sorted(set(np.bincount(Z, minlength=16).tolist()))\n    # SBZ: S(h ⊕ S(n)) — n faol (h doimiy), h faol (n doimiy), ikkalasi bir xil faol (h = n = x)\n    R[\'SBZ_n_faol_turli\'] = int(len(np.unique(S[0x3c ^ S[x]]))); R[\'SBZ_h_faol_turli\'] = int(len(np.unique(S[x ^ S[0x5a]])))\n    R[\'SBZ_ikkalasi_faol_turli\'] = int(len(np.unique(S[x ^ S[x]]))); R[\'SBZ_ikkalasi_xor\'] = int(np.bitwise_xor.reduce(S[x ^ S[x]]))\n    # Massiv darajasida: bitta faol bayt (holat[0,0]) — Qo\'sh, Aralash, BZ ning ta\'siri\n    for p in (256, 16):\n        c = Cfg(p, bz); N = p; acc = {\'Qosh_holat\': [], \'Qosh_holatn\': [], \'Aralash\': [], \'BZ_n_faol\': [], \'XOR_n_faol\': [], \'BZ_h_faol\': []}\n        for _ in range(trials // 4):\n            k = kalitlar(rng, 1, p)[0]; h = rng.integers(0, p, 32); hn = rng.integers(0, p, 32)\n            H = np.tile(h, (N, 1)); H[:, 0] = np.arange(N); HN = np.tile(hn, (N, 1)); K = np.tile(k, (N, 1))\n            a, b = qosh(H, HN, K[:, 0], c); acc[\'Qosh_holat\'].append(sana(xossa(a, p))); acc[\'Qosh_holatn\'].append(sana(xossa(b, p)))\n            acc[\'Aralash\'].append(sana(xossa(aralash(H, K, c), p)))\n            HN2 = HN.copy(); HN2[:, 0] = np.arange(N)\n            if p == 256 or bz == \'sbz\': acc[\'BZ_n_faol\'].append(sana(xossa(bzf(np.tile(h, (N, 1)), HN2, c), p)))\n            else: acc[\'XOR_n_faol\'].append(sana(xossa(bzf(np.tile(h, (N, 1)), HN2, c), p)))\n            acc[\'BZ_h_faol\'].append(sana(xossa(bzf(H, np.tile(hn, (N, 1)), c), p)))\n            cs = Cfg(p, \'sbz\'); acc.setdefault(\'SBZ_n_faol\', []).append(sana(xossa(bzf(np.tile(h, (N, 1)), HN2, cs), p)))\n        R[f\'massiv_{p}\'] = {k_: {t: round(float(np.mean([d[t] for d in v])), 2) for t in (\'A\', \'C\', \'B\', \'B7\', \'U\')} for k_, v in acc.items() if v}\n    return R\n\ndef raund_kuzatish(rng, p, joy, T=40, bz=\'std\'):\n    """joy=\'holat\': 1-raund kirishidagi holat[0,0] faol; joy=\'blok\': kirish blokining 0-elementi faol (0-nuqtadan boshlab).\n    Natija: har akslantirishdan keyin har pozitsiyaning xossasi; T ta tajriba bo‘yicha \'barcha tajribalarda B yoki A\' ulushi."""\n    c = Cfg(p, bz); N = p; rec = {}; sums = {}\n    nb = 8 if p == 256 else 4\n    for _ in range(T):\n        k = kalitlar(rng, 1, p)[0]; h = rng.integers(0, p, 32); bl = rng.integers(0, p, 32)\n        H = np.tile(h, (N, 1)); BL = np.tile(bl, (N, 1)); K = np.tile(k, (N, 1))\n        if joy == \'holat\': H[:, 0] = np.arange(N)\n        else: BL[:, 0] = np.arange(N)\n        tr = []\n        if joy == \'holat\':\n            from differensial import raundlar_yutishsiz\n            raundlar_yutishsiz(H, BL, K.copy(), K, c, tr)\n        else:\n            iteration(H, BL, K.copy(), K, c, trace=tr)\n        for nm, j, hh, kk in tr:\n            key = f\'{j}:{nm}\'; rec.setdefault(key, []).append(xossa(hh, p))\n            sums.setdefault(key, []).append(np.bitwise_xor.reduce(hh, axis=0))\n    out = []\n    for key, lst in rec.items():\n        j, nm = key.split(\':\'); arr = np.array(lst)          # (T, 32)\n        bal_all = ((arr == \'A\') | (arr == \'B\') | (arr == \'C\')).all(0)   # doim balanslangan (XOR-yig‘indi 0)\n        b7_all = ((arr == \'A\') | (arr == \'B\') | (arr == \'C\') | (arr == \'B7\')).all(0)\n        SS = np.array(sums[key])                              # (T, 32)\n        qatlam = [int((((SS >> b) & 1) == 0).all(0).sum()) for b in range(nb)]\n        out.append(dict(raund=int(j), amal=nm, doim_B=int(bal_all.sum()), doim_B7=int(b7_all.sum()), qatlam_doim_nol=qatlam,\n                        ortacha=sana([t for row in lst for t in row]), T=T))\n        out[-1][\'ortacha\'] = {t: round(v / T, 2) for t, v in out[-1][\'ortacha\'].items()}\n    return out\n\ndef toliq_xesh(rng, T=40, bz=\'std\'):\n    """Standart-128: bir blokli xabar, 0-bayt faol (256 xabar). Standart-256: 2 blokli xabar, 2-blokning 0-bayti faol."""\n    R = {}\n    for p, L in ((16, 16), (256, 64)):\n        c = Cfg(p, bz); zer = np.zeros((T, 32, 8 if p == 256 else 4), dtype=bool)\n        for t in range(T):\n            k = kalitlar(rng, 1, 256)[0] if p == 256 else rng.integers(0, 256, 16)\n            if p == 16: k[0] &= 0xEE\n            m = rng.integers(0, 256, L); Mm = np.tile(m, (256, 1)); pos = 0 if p == 16 else 32\n            Mm[:, pos] = np.arange(256)\n            out = hash_msgs(Mm, np.tile(k, (256, 1)), c); s = np.bitwise_xor.reduce(out, axis=0)\n            zer[t] = np.array([(s >> j) & 1 for j in range(zer.shape[2])]).T == 0\n        R[f\'{p}\'] = {\'qatlam_boyicha_doim_nol_bitlar\': [int(zer[:, :, j].all(0).sum()) for j in range(zer.shape[2])],\n                     \'qatlam_boyicha_nol_ulushi\': [round(float(zer[:, :, j].mean()), 3) for j in range(zer.shape[2])]}\n    return R\n\ndef tfunksiya_128(rng, N=2000, bz=\'std\'):\n    """Standart-128, bir blokli xabar: faqat yuqori (3-) bit-qatlamdagi w ta bit o‘zgartirilganda\n    chiqishning 0–2-qatlamlari (96 bit) saqlanish ehtimolligi."""\n    c = Cfg(16, bz); R = {}\n    for w in (1, 2, 4, 8, 32):\n        k = rng.integers(0, 256, (N, 16)); k[:, 0] &= 0xEE\n        m = rng.integers(0, 256, (N, 16)); u = to_units(m, 16); u2 = u.copy()\n        for n in range(N):\n            pos = rng.choice(32, w, replace=False); u2[n, pos] ^= 8\n        m2 = np.array([[u2[n, 2*i] << 4 | u2[n, 2*i+1] for i in range(16)] for n in range(N)])\n        o1 = hash_msgs(m, k, c); o2 = hash_msgs(m2, k, c)\n        low = ((o1 ^ o2) & 7) == 0\n        R[str(w)] = {\'quyi_3_qatlam_saqlandi\': round(float(low.all(1).mean()), 4), \'quyi_qatlam_bitlari_saqlandi\': round(float(low.mean()), 4),\n                     \'0_qatlam_saqlandi\': round(float((((o1 ^ o2) & 1) == 0).all(1).mean()), 4)}\n    return R\n\nif __name__ == "__main__":\n    ap = argparse.ArgumentParser(); ap.add_argument(\'--tez\', action=\'store_true\')\n    ap.add_argument(\'--bz\', default=\'std\', choices=[\'std\', \'sbz\'], help="yutish o‘zgartirishi: std (BaytZichlash) yoki sbz")\n    a = ap.parse_args(); s = 0.25 if a.tez else 1.0\n    os.makedirs(\'natijalar\', exist_ok=True); out = {}\n    print("1. Akslantirishlarning to‘plamlarga ta’siri..."); out[\'komponentlar\'] = komponentlar(np.random.default_rng(41), int(200 * s), a.bz)\n    print("2. Raundlar bo‘yicha kuzatish...")\n    for p in (256, 16):\n        for joy in (\'holat\', \'blok\'):\n            out[f\'raund_{p}_{joy}\'] = raund_kuzatish(np.random.default_rng(42), p, joy, int(40 * s), a.bz); print(f"   p={p}, {joy}: tayyor")\n    print("3. To‘liq xesh..."); out[\'toliq_xesh\'] = toliq_xesh(np.random.default_rng(43), int(40 * s), a.bz)\n    print("4. 128 bitli rejimda yuqori bit-qatlam o‘zgarishi..."); out[\'tfunksiya_128\'] = tfunksiya_128(np.random.default_rng(44), int(2000 * s), a.bz)\n    fn = \'natijalar/integral.json\' if a.bz == \'std\' else \'natijalar/integral_sbz.json\'\n    json.dump(out, open(fn, \'w\'), indent=1, ensure_ascii=False)\n    print(\'Saqlandi:\', fn)\n',
}
for _nom, _matn in _MODULLAR.items():
    if importlib.util.find_spec(_nom) is not None and _nom not in sys.modules:
        print(f"{_nom}.py — papkadagi fayldan olinadi"); continue
    _m = types.ModuleType(_nom); _m.__file__ = f"<{_nom}>"
    sys.modules[_nom] = _m; exec(compile(_matn, f"<{_nom}>", "exec"), _m.__dict__)
    print(f"{_nom} — daftar ichidan yuklandi")


algoritm — daftar ichidan yuklandi
komponentlar — daftar ichidan yuklandi
differensial — daftar ichidan yuklandi
algebraik — daftar ichidan yuklandi
integral — daftar ichidan yuklandi


In [2]:
import numpy as np, itertools, math, json, collections
from algoritm import *
from komponentlar import *
import algebraik, integral
from differensial import raundlar_yutishsiz
HAJM = 1          # 1 — tez; kattaroq qiymat tanlanmalarni oshiradi
OGIR = False      # True — to‘liq xesh uchun 2^16 to‘plamli integral sinov
R = {}
print("Nazorat qiymatlari:", "hammasi mos" if nazorat() else "XATO")

  p=256, std: MOS
  p=256, sbz: MOS
  p= 16, std: MOS
  p= 16, sbz: MOS
Nazorat qiymatlari: hammasi mos


## 1. Chiziqli kriptotahlil
### 1.1. S va SBZ korrelyatsion matritsalari

SBZ uchun c_SBZ((α, β) → γ) = c_S(α → γ) · c_S(β → α). To‘liq taqsimot 255 × 65536 element bo‘yicha hisoblanadi.

In [3]:
WS = lat(S, 8); A = np.abs(WS)
v, c = np.unique(A[1:, :], return_counts=True)
R['S_lat'] = {int(a): int(b) for a, b in zip(v, c)}
print("S: |W| taqsimoti:", R['S_lat'], " maks |c| =", A[1:, :].max() / 256)
cnt = collections.Counter()
for g in range(1, 256):
    for a in range(1, 256):
        if A[g, a]:
            vv, cc = np.unique(A[g, a] * A[a, :], return_counts=True)
            for x, y in zip(vv, cc): cnt[int(x)] += int(y)
cnt[0] = 255 * 65536 - sum(cnt.values())
R['SBZ_lat'] = dict(sorted(cnt.items()))
print("SBZ: maks |c| =", max(k for k in cnt if cnt[k]) / 65536, "= 2^-6;  bunday elementlar soni:", cnt[1024])
print("γ = 01 uchun eng yaxshi approksimatsiyalar (α, β, c):")
for a in np.argsort(-A[1])[:6]:
    b = int(np.argmax(A[a])); print(f"   α = {a:02X}, β = {b:02X}, c = {WS[1, a] * WS[a, b] / 65536:+.6f}")
hh = np.arange(65536) >> 8; nn = np.arange(65536) & 255; y = S[hh ^ S[nn]]
rel = (y & 1) ^ PAR8[hh & 0x2D] ^ PAR8[nn & 0x1A]
print("Tekshiruv: y0 ⊕ <2D,h> ⊕ <1A,n> = 1 ifodasi", int(rel.sum()), "/ 65536 juftlikda 1 ga teng, c =", 1 - 2 * rel.sum() / 65536)

S: |W| taqsimoti: {0: 4335, 4: 12240, 8: 9180, 12: 10200, 16: 8670, 20: 6120, 24: 9180, 28: 4080, 32: 1275}  maks |c| = 0.125
SBZ: maks |c| = 0.015625 = 2^-6;  bunday elementlar soni: 6375
γ = 01 uchun eng yaxshi approksimatsiyalar (α, β, c):
   α = 2D, β = 1A, c = -0.015625
   α = 67, β = 05, c = -0.015625
   α = A3, β = 06, c = +0.015625
   α = 8E, β = 0A, c = +0.015625
   α = C4, β = 13, c = +0.015625
   α = 3B, β = 1D, c = +0.013672
Tekshiruv: y0 ⊕ <2D,h> ⊕ <1A,n> = 1 ifodasi 33280 / 65536 juftlikda 1 ga teng, c = -0.015625


### 1.2. 0-qatlam raund tenglamasi h₀ ← L·h₀ va yutish qatlamlari bo‘yicha chegara

In [4]:
rng = np.random.default_rng(7); R['lsb'] = {}
for p in (256, 16):
    c = Cfg(p, 'sbz'); N = 300 * HAJM; k = kalitlar(rng, N, p)
    h = rng.integers(0, p, (N, 32)); bl = rng.integers(0, p, (N, 32)); tr = []
    iteration(h, bl, k.copy(), k, c, trace=tr)
    cur = [t for t in tr if t[0] == 'BZ0'][0][2] % 2; Ls = [L_raund(k[n]) for n in range(N)]; cl = []
    for j in range(1, 11):
        cur = np.array([(Ls[n] @ cur[n]) % 2 for n in range(N)]); act = [t for t in tr if t[1] == j][-1][2] % 2
        cl.append(round(2 * float((cur == act).mean()) - 1, 3))
    R['lsb'][p] = cl; print(f"SBZ-{p if p == 256 else 128}: 0-qatlam tenglamasi korrelyatsiyasi (raund 1..10): {cl}")
def zanjir(Lm):
    L8 = np.linalg.matrix_power(Lm, 8) % 2; maps = [L8, Lm, Lm, L8, Lm]; best = {k: 99 for k in range(1, 7)}
    cands = [np.eye(32, dtype=np.int64)[list(c)].sum(0) % 2 for w in (1, 2) for c in itertools.combinations(range(32), w)]
    for end in range(1, 6):
        for u in cands:
            tot = int(u.sum()); cur = u; n = 1; best[1] = min(best[1], tot)
            for m in range(end - 1, -1, -1):
                cur = (maps[m].T @ cur) % 2
                if cur.sum() == 0: break
                tot += int(cur.sum()); n += 1; best[n] = min(best[n], tot)
    return best
zs = {k: [] for k in range(1, 7)}
for t in range(15 * HAJM):
    zz = zanjir(L_raund(kalitlar(rng, 1, 256)[0]))
    for k_ in zz:
        if zz[k_] < 99: zs[k_].append(zz[k_])
R['zanjir'] = {k_: (min(v), float(np.median(v))) for k_, v in zs.items()}
print("Qatlamlar | faol baytlar min / mediana | SBZ chegarasi log2|c| (mediana / eng yomon)")
for k_, (mn, md_) in R['zanjir'].items(): print(f"   {k_}   |   {mn} / {md_}   |   ≤ {-6*md_:.0f} / ≤ {-6*mn}")

SBZ-256: 0-qatlam tenglamasi korrelyatsiyasi (raund 1..10): [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.001, 0.012, 0.082]
SBZ-128: 0-qatlam tenglamasi korrelyatsiyasi (raund 1..10): [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.01, 0.026, 0.083]
Qatlamlar | faol baytlar min / mediana | SBZ chegarasi log2|c| (mediana / eng yomon)
   1   |   1 / 1.0   |   ≤ -6 / ≤ -6
   2   |   2 / 2.0   |   ≤ -12 / ≤ -12
   3   |   3 / 3.0   |   ≤ -18 / ≤ -18
   4   |   4 / 8.0   |   ≤ -48 / ≤ -24
   5   |   5 / 16.0   |   ≤ -96 / ≤ -30
   6   |   6 / 24.0   |   ≤ -144 / ≤ -36


## 2. Differensial kriptotahlil
### 2.1. SBZ ayirmalar matritsasi (65536 × 256, to‘liq)

In [5]:
DS = ddt(S, 256).astype(np.float64); idx = np.arange(256); dist = collections.Counter(); mx = (0, None); son = 0
for dn in range(256):
    Rm = (DS[dn][idx[:, None] ^ idx[None, :]] @ DS).astype(np.int64) if dn else DS.astype(np.int64) * 256
    v, c = np.unique(Rm, return_counts=True); dist.update({int(a): int(b) for a, b in zip(v, c)})
    sub = Rm.copy()
    if dn == 0: sub[0, :] = 0
    if sub.max() > mx[0]: mx = (int(sub.max()), (dn,) + tuple(int(t) for t in np.unravel_index(sub.argmax(), sub.shape)))
    if dn: son += int(Rm[0, 0])
R['SBZ_ddt_max'] = mx[0]
print("Maks. ehtimollik:", mx[0], "/ 65536 =", mx[0] / 65536, " (Δn, Δh, Δy) =", mx[1])
print("Δn ≠ 0 dagi qiymatlar oralig‘i:", min(k for k in dist if 0 < k < 512), "…", max(k for k in dist if 0 < k < 512), "(×2^-16)")
print("Δh = 0, Δn ≠ 0 da so‘nishlar soni:", son)

Maks. ehtimollik: 1024 / 65536 = 0.015625  (Δn, Δh, Δy) = (0, 1, 31)
Δn ≠ 0 dagi qiymatlar oralig‘i: 164 … 340 (×2^-16)
Δh = 0, Δn ≠ 0 da so‘nishlar soni: 0


### 2.2. Orttirmalarning raundlar orqali o‘tishi (yutishdan keyingi holatda bitta MSB ayirmasi)

In [6]:
rng = np.random.default_rng(9); R['msb'] = {}
for p in (256, 16):
    c = Cfg(p, 'sbz'); N = 500 * HAJM; msb = p // 2; k = kalitlar(rng, N, p)
    h = rng.integers(0, p, (N, 32)); bl = rng.integers(0, p, (N, 32)); h2 = h.copy(); h2[:, 0] ^= msb
    t1, t2 = [], []; raundlar_yutishsiz(h, bl, k.copy(), k, c, t1); raundlar_yutishsiz(h2, bl, k.copy(), k, c, t2)
    rows = [(a[1], round(float((((a[2] ^ b[2]) == 0) | ((a[2] ^ b[2]) == msb)).all(1).mean()), 3), round(float(((a[2] ^ b[2]) != 0).sum(1).mean()), 1))
            for a, b in zip(t1, t2) if a[0] == 'Tuz']
    R['msb'][p] = rows; print(f"SBZ-{p if p == 256 else 128}: (raund, P(MSB izi), faol elementlar):", rows)

SBZ-256: (raund, P(MSB izi), faol elementlar): [(1, 0.944, 7.0), (2, 0.822, 13.0), (3, 0.698, 17.6), (4, 0.586, 20.7), (5, 0.504, 22.3), (6, 0.41, 24.1), (7, 0.344, 25.4), (8, 0.038, 27.4), (9, 0.038, 30.7), (10, 0.038, 30.7)]
SBZ-128: (raund, P(MSB izi), faol elementlar): [(1, 0.494, 13.9), (2, 0.166, 22.4), (3, 0.086, 25.0), (4, 0.046, 26.2), (5, 0.038, 26.4), (6, 0.036, 26.6), (7, 0.036, 26.7), (8, 0.036, 28.9), (9, 0.036, 28.5), (10, 0.036, 28.6)]


## 3. Algebraik kriptotahlil

In [7]:
x = np.arange(256)
print("S: kvadrat implitsit tenglamalar:", implicit_count(np.concatenate([bits(x, 8), bits(S[x], 8)], 1), 2))
hh = np.arange(65536) >> 8; nn = np.arange(65536) & 255; y = S[hh ^ S[nn]]
print("SBZ: kvadrat implitsit tenglamalar (24 o‘zgaruvchi):", implicit_count(np.concatenate([bits(hh, 8), bits(nn, 8), bits(y, 8)], 1), 2))
print("SBZ: chiqish bitlari darajalari:", [anf_deg((y >> j) & 1, 16) for j in range(8)])
R['kub'] = algebraik.kub_testlari(np.random.default_rng(21), dlar=(4, 8, 10), raundlar=(0, 1)) if HAJM >= 2 else None
rng = np.random.default_rng(31); R['lin'] = algebraik.linearizatsiya_hujumi(rng, 80, 100 * HAJM)

S: kvadrat implitsit tenglamalar: (39, 137)
SBZ: kvadrat implitsit tenglamalar (24 o‘zgaruvchi): (0, 301)
SBZ: chiqish bitlari darajalari: [13, 13, 13, 13, 13, 13, 13, 13]
  p=16, std: {'toliq_32_bit': 1.0, 'bitlar_boyicha': 1.0}
  p=16, sbz: {'toliq_32_bit': 0.0, 'bitlar_boyicha': 0.5616}
  p=256, std: {'toliq_32_bit': 0.0, 'bitlar_boyicha': 0.5047}
  p=256, sbz: {'toliq_32_bit': 0.0, 'bitlar_boyicha': 0.6422}


## 4. Integral kriptotahlil

In [8]:
R['int_komp'] = integral.komponentlar(np.random.default_rng(41), 40 * HAJM, 'sbz')
print("SBZ: n faol — turli qiymatlar", R['int_komp'].get('SBZ_n_faol_turli'), "; h faol —", R['int_komp'].get('SBZ_h_faol_turli'))
R['int_raund'] = {}
for p in (256, 16):
    for joy in ('holat', 'blok'):
        rr = integral.raund_kuzatish(np.random.default_rng(42), p, joy, 8 * HAJM, 'sbz'); R['int_raund'][f'{p}_{joy}'] = rr
        print(f"SBZ-{p if p == 256 else 128}, faol {joy}: raundlar oxirida 0-qatlamda doim balanslangan bitlar:",
              [r['qatlam_doim_nol'][0] for r in rr if r['amal'] == 'Tuz'])
R['int_toliq'] = integral.toliq_xesh(np.random.default_rng(43), 8 * HAJM, 'sbz')
print("To‘liq xesh (bitta faol bayt), bit-qatlamlar bo‘yicha doim nol bitlar:", {k: v['qatlam_boyicha_doim_nol_bitlar'] for k, v in R['int_toliq'].items()})

SBZ: n faol — turli qiymatlar 256 ; h faol — 256
SBZ-256, faol holat: raundlar oxirida 0-qatlamda doim balanslangan bitlar: [32, 32, 32, 32, 32, 32, 32, 0, 0, 0]
SBZ-256, faol blok: raundlar oxirida 0-qatlamda doim balanslangan bitlar: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
SBZ-128, faol holat: raundlar oxirida 0-qatlamda doim balanslangan bitlar: [32, 32, 32, 32, 32, 32, 32, 0, 1, 0]
SBZ-128, faol blok: raundlar oxirida 0-qatlamda doim balanslangan bitlar: [2, 0, 1, 1, 0, 0, 0, 1, 0, 0]
To‘liq xesh (bitta faol bayt), bit-qatlamlar bo‘yicha doim nol bitlar: {'16': [0, 0, 0, 0], '256': [0, 0, 0, 1, 0, 0, 0, 0]}


### 4.1. Yuqori tartibli integral: 2¹⁶ elementli to‘plam (blokning ikki bayti faol)

In [9]:
def ikki_bayt_iteratsiya(p, T):
    c = Cfg(p, 'sbz'); rng = np.random.default_rng(5); nb = 8 if p == 256 else 4; acc = []
    for t in range(T):
        k = kalitlar(rng, 1, p)[0]; h = rng.integers(0, p, 32); bl = rng.integers(0, p, 32); N = 65536
        H = np.tile(h, (N, 1)); BL = np.tile(bl, (N, 1))
        if p == 256: BL[:, 0] = np.arange(N) >> 8; BL[:, 1] = np.arange(N) & 255
        else:
            for i in range(4): BL[:, i] = (np.arange(N) >> (12 - 4 * i)) & 15
        K = np.tile(k, (N, 1)); tr = []; iteration(H, BL, K.copy(), K, c, trace=tr, rounds=3)
        acc.append([np.array([((np.bitwise_xor.reduce(hh_, axis=0) >> b) & 1) == 0 for b in range(nb)]) for nm, j, hh_, kk in tr if nm in ('BZ0', 'Tuz')])
    return [np.array([a[i] for a in acc]).all(0).sum(1).tolist() for i in range(len(acc[0]))]
for p in (256, 16):
    r = ikki_bayt_iteratsiya(p, 2 * HAJM); R[f'ikki_bayt_{p}'] = r
    print(f"SBZ-{p if p == 256 else 128}: SBZ(0) dan keyin va 1–3-raundlar oxirida doim nol bitlar (qatlamlar bo‘yicha):", r)
if OGIR:
    def toliq_ikki_bayt(p, L, act, T):
        c = Cfg(p, 'sbz'); rng = np.random.default_rng(11); nb = 8 if p == 256 else 4; z = []
        for t in range(T):
            k = kalitlar(rng, 1, 256)[0] if p == 256 else rng.integers(0, 256, 16)
            if p == 16: k[0] &= 0xEE
            m = rng.integers(0, 256, L); N = 65536; M = np.tile(m, (N, 1)); M[:, act[0]] = np.arange(N) >> 8; M[:, act[1]] = np.arange(N) & 255
            sm = np.zeros(32, dtype=np.int64)
            for s0 in range(0, N, 16384): sm ^= np.bitwise_xor.reduce(hash_msgs(M[s0:s0 + 16384], np.tile(k, (min(16384, N - s0), 1)), c), axis=0)
            z.append(np.array([((sm >> b) & 1) == 0 for b in range(nb)]))
        return np.array(z).all(0).sum(1).tolist()
    R['toliq_ikki_bayt_128'] = toliq_ikki_bayt(16, 16, (0, 1), 2); print("SBZ-128 to‘liq xesh, 2^16 to‘plam:", R['toliq_ikki_bayt_128'])
json.dump(R, open('sbz_natijalari.json', 'w'), ensure_ascii=False, indent=1, default=lambda o: o.tolist() if hasattr(o, 'tolist') else str(o))
print("Natijalar sbz_natijalari.json ga yozildi.")

SBZ-256: SBZ(0) dan keyin va 1–3-raundlar oxirida doim nol bitlar (qatlamlar bo‘yicha): [[30, 31, 30, 31, 30, 30, 30, 32], [28, 9, 10, 11, 8, 5, 8, 3], [18, 8, 9, 6, 5, 6, 6, 6], [11, 11, 10, 9, 10, 12, 10, 8]]
SBZ-128: SBZ(0) dan keyin va 1–3-raundlar oxirida doim nol bitlar (qatlamlar bo‘yicha): [[32, 32, 32, 32], [32, 11, 6, 6], [32, 11, 9, 12], [32, 12, 9, 6]]
Natijalar sbz_natijalari.json ga yozildi.
